# 🤖 AI Generator — Telegram Bot (видео до 10 сек)
### Инструкция:
1. Вставь токен в Шаге 1
2. Запусти все ячейки по порядку
3. Открой бота в Telegram и напиши /start

> ⚠️ Включи GPU: `Среда выполнения → Сменить → T4 GPU`

In [ ]:
# ШАГ 1 — Вставь токен от @BotFather
BOT_TOKEN = '8586682626:AAGR8P3U910Y6ftImVwbVfX7e35LBLlFynI'

# Настройки видео
VIDEO_CLIPS   = 3    # Сколько клипов склеивать (3 = ~9 сек, 2 = ~6 сек)
FRAMES_CLIP   = 24   # Кадров на клип (максимум для ModelScope)
FPS           = 8    # Кадров в секунду
# Итого: VIDEO_CLIPS * FRAMES_CLIP / FPS = секунд
print(f'⏱ Длина видео: {VIDEO_CLIPS * FRAMES_CLIP / FPS:.0f} секунд')

In [ ]:
# ШАГ 2 — Установка зависимостей
import subprocess

print('📦 1/5: Фиксирую httpx...')
subprocess.run('pip install -q "httpx>=0.28.1,<1.0.0" --upgrade', shell=True)
print('📦 2/5: diffusers...')
subprocess.run('pip install -q diffusers transformers accelerate xformers', shell=True)
print('📦 3/5: медиа-библиотеки...')
subprocess.run('pip install -q imageio imageio-ffmpeg Pillow', shell=True)
print('📦 4/5: moviepy (склейка видео)...')
subprocess.run('pip install -q moviepy', shell=True)
print('📦 5/5: Telegram + переводчик...')
subprocess.run('pip install -q "python-telegram-bot==20.7" deep-translator', shell=True)

import httpx
print(f'\n✅ httpx: {httpx.__version__}')
print('✅ Все зависимости установлены!')

In [ ]:
# ШАГ 3 — Загрузка моделей
import torch, gc, os
from diffusers import StableDiffusionXLPipeline, DiffusionPipeline, DPMSolverMultistepScheduler
from diffusers.utils import export_to_video

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️ Устройство: {device}')
if device == 'cuda':
    print(f'🎮 {torch.cuda.get_device_name(0)}')
else:
    print('⚠️ GPU не найден! Включи T4 в настройках Colab.')

print('\n⏳ Загрузка SDXL (~5 мин)...')
image_pipe = StableDiffusionXLPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0',
    torch_dtype=torch.float16,
    use_safetensors=True,
    variant='fp16'
).to(device)
image_pipe.scheduler = DPMSolverMultistepScheduler.from_config(image_pipe.scheduler.config)
try:
    image_pipe.enable_xformers_memory_efficient_attention()
except:
    pass
print('✅ SDXL загружен!')

print('\n⏳ Загрузка видео-модели (~5 мин)...')
video_pipe = DiffusionPipeline.from_pretrained(
    'ali-vilab/text-to-video-ms-1.7b',
    torch_dtype=torch.float16,
    variant='fp16'
).to(device)
video_pipe.enable_model_cpu_offload()
video_pipe.enable_vae_slicing()
print('✅ Видео-модель загружена!')

os.makedirs('/content/outputs', exist_ok=True)
print('\n🚀 Все модели готовы!')

In [ ]:
# ШАГ 4 — Функция склейки видео
import imageio
import numpy as np
from moviepy.editor import VideoFileClip, concatenate_videoclips

def generate_long_video(prompt, negative_prompt, num_clips, frames_per_clip, fps, output_path):
    """Генерирует несколько клипов и склеивает их в одно видео."""
    clip_paths = []

    for i in range(num_clips):
        print(f'  🎬 Клип {i+1}/{num_clips}...')
        result = video_pipe(
            prompt=prompt,
            negative_prompt=negative_prompt,
            num_frames=frames_per_clip,
            num_inference_steps=25,
            # Разный seed для каждого клипа — чуть разное движение
            generator=torch.Generator().manual_seed(i * 42)
        )
        clip_path = f'{output_path}_clip_{i}.mp4'
        export_to_video(result.frames[0], clip_path, fps=fps)
        clip_paths.append(clip_path)
        gc.collect()
        torch.cuda.empty_cache()

    print('  ✂️ Склеиваю клипы...')
    clips = [VideoFileClip(p) for p in clip_paths]
    final = concatenate_videoclips(clips, method='compose')
    final.write_videofile(output_path, codec='libx264', audio=False, verbose=False, logger=None)

    # Удаляем временные клипы
    for p in clip_paths:
        try:
            os.remove(p)
        except:
            pass

    duration = num_clips * frames_per_clip / fps
    print(f'  ✅ Готово! Длина: {duration:.1f} сек')
    return output_path

print('✅ Функция склейки готова!')

In [ ]:
# ШАГ 5 — Запуск Telegram бота
import gc
from telegram import Update, InlineKeyboardButton, InlineKeyboardMarkup
from telegram.ext import Application, CommandHandler, MessageHandler, CallbackQueryHandler, filters, ContextTypes
from deep_translator import GoogleTranslator
import torch

user_state = {}

STYLES = {
    'photo':      ('📷 Фото',      'photorealistic, ultra detailed, 4k, sharp focus, professional photography'),
    'anime':      ('⛩ Аниме',     'anime style, manga, vibrant colors, Studio Ghibli, detailed illustration'),
    'oil':        ('🎨 Масло',     'oil painting, classical art, textured brushstrokes, museum quality'),
    'cyber':      ('🌆 Киберпанк', 'cyberpunk, neon lights, futuristic city, blade runner, dark atmosphere'),
    'watercolor': ('💧 Акварель',  'watercolor painting, soft colors, artistic, delicate brushwork, dreamy'),
    'fantasy':    ('🐉 Фэнтези',  'fantasy art, epic, magical, dramatic lighting, concept art, artstation'),
    'render3d':   ('🧊 3D',        '3D render, octane render, hyperrealistic, cinema4d, subsurface scattering'),
    'pixel':      ('👾 Пиксель',   'pixel art, 8-bit, retro game style, pixelated, sprite art'),
}

def translate_to_en(text):
    try:
        return GoogleTranslator(source='auto', target='en').translate(text)
    except:
        return text

def get_state(uid):
    if uid not in user_state:
        user_state[uid] = {
            'mode': 'image', 'style': 'photo',
            'steps': 30, 'cfg': 7.5,
            'video_clips': VIDEO_CLIPS  # из Шага 1
        }
    return user_state[uid]

async def start(update: Update, ctx: ContextTypes.DEFAULT_TYPE):
    get_state(update.effective_user.id)
    duration = VIDEO_CLIPS * FRAMES_CLIP / FPS
    await update.message.reply_text(
        '🎨 *AI Generator Bot*\n\n'
        'Напиши что сгенерировать — на русском или английском!\n\n'
        '*Команды:*\n'
        '🖼 /image — режим изображений\n'
        f'🎬 /video — режим видео (~{duration:.0f} сек)\n'
        '🎨 /style — выбрать стиль\n'
        '⚙️ /settings — настройки\n'
        '⏱ /duration — длина видео\n'
        'ℹ️ /status — текущие настройки\n\n'
        '_Пример: закат над морем с туманом_',
        parse_mode='Markdown'
    )

async def set_image(update: Update, ctx: ContextTypes.DEFAULT_TYPE):
    get_state(update.effective_user.id)['mode'] = 'image'
    await update.message.reply_text('🖼 Режим: *Изображение*\nПиши промпт!', parse_mode='Markdown')

async def set_video(update: Update, ctx: ContextTypes.DEFAULT_TYPE):
    s = get_state(update.effective_user.id)
    s['mode'] = 'video'
    duration = s['video_clips'] * FRAMES_CLIP / FPS
    await update.message.reply_text(
        f'🎬 Режим: *Видео* (~{duration:.0f} сек)\nПиши промпт!',
        parse_mode='Markdown'
    )

async def duration_menu(update: Update, ctx: ContextTypes.DEFAULT_TYPE):
    keyboard = [
        [InlineKeyboardButton(f'⚡ ~3 сек (1 клип)',  callback_data='clips_1'),
         InlineKeyboardButton(f'▶️ ~6 сек (2 клипа)', callback_data='clips_2')],
        [InlineKeyboardButton(f'🎬 ~9 сек (3 клипа)', callback_data='clips_3'),
         InlineKeyboardButton(f'🎥 ~12 сек (4 клипа)', callback_data='clips_4')],
    ]
    s = get_state(update.effective_user.id)
    cur = s['video_clips'] * FRAMES_CLIP / FPS
    await update.message.reply_text(
        f'⏱ *Длина видео*\nСейчас: ~{cur:.0f} сек\n\n'
        f'⚠️ Больше клипов = дольше ждать\n(каждый клип ~3-5 мин)',
        reply_markup=InlineKeyboardMarkup(keyboard),
        parse_mode='Markdown'
    )

async def duration_callback(update: Update, ctx: ContextTypes.DEFAULT_TYPE):
    query = update.callback_query
    await query.answer()
    clips = int(query.data.replace('clips_', ''))
    s = get_state(query.from_user.id)
    s['video_clips'] = clips
    duration = clips * FRAMES_CLIP / FPS
    await query.edit_message_text(
        f'✅ Длина видео: *~{duration:.0f} сек* ({clips} клип(а))\nПиши промпт!',
        parse_mode='Markdown'
    )

async def style_menu(update: Update, ctx: ContextTypes.DEFAULT_TYPE):
    keyboard = [
        [InlineKeyboardButton(STYLES['photo'][0],      callback_data='style_photo'),
         InlineKeyboardButton(STYLES['anime'][0],      callback_data='style_anime')],
        [InlineKeyboardButton(STYLES['oil'][0],        callback_data='style_oil'),
         InlineKeyboardButton(STYLES['cyber'][0],      callback_data='style_cyber')],
        [InlineKeyboardButton(STYLES['watercolor'][0], callback_data='style_watercolor'),
         InlineKeyboardButton(STYLES['fantasy'][0],    callback_data='style_fantasy')],
        [InlineKeyboardButton(STYLES['render3d'][0],   callback_data='style_render3d'),
         InlineKeyboardButton(STYLES['pixel'][0],      callback_data='style_pixel')],
    ]
    await update.message.reply_text('🎨 Выбери стиль:', reply_markup=InlineKeyboardMarkup(keyboard))

async def style_callback(update: Update, ctx: ContextTypes.DEFAULT_TYPE):
    query = update.callback_query
    await query.answer()
    style_key = query.data.replace('style_', '')
    get_state(query.from_user.id)['style'] = style_key
    await query.edit_message_text(f'✅ Стиль: *{STYLES[style_key][0]}*\nПиши промпт!', parse_mode='Markdown')

async def settings_menu(update: Update, ctx: ContextTypes.DEFAULT_TYPE):
    s = get_state(update.effective_user.id)
    keyboard = [
        [InlineKeyboardButton('🐢 Быстро (20)', callback_data='steps_20'),
         InlineKeyboardButton('⚖️ Баланс (30)', callback_data='steps_30'),
         InlineKeyboardButton('💎 Качество (50)', callback_data='steps_50')],
        [InlineKeyboardButton('🌊 CFG 5',   callback_data='cfg_5'),
         InlineKeyboardButton('⚖️ CFG 7.5', callback_data='cfg_7.5'),
         InlineKeyboardButton('🎯 CFG 12',  callback_data='cfg_12')],
    ]
    await update.message.reply_text(
        f'⚙️ *Настройки*\nШаги: {s["steps"]} | CFG: {s["cfg"]}',
        reply_markup=InlineKeyboardMarkup(keyboard),
        parse_mode='Markdown'
    )

async def settings_callback(update: Update, ctx: ContextTypes.DEFAULT_TYPE):
    query = update.callback_query
    await query.answer()
    s = get_state(query.from_user.id)
    if query.data.startswith('steps_'):
        s['steps'] = int(query.data.replace('steps_', ''))
    elif query.data.startswith('cfg_'):
        s['cfg'] = float(query.data.replace('cfg_', ''))
    await query.edit_message_text(
        f'✅ Шаги: *{s["steps"]}* | CFG: *{s["cfg"]}*',
        parse_mode='Markdown'
    )

async def status(update: Update, ctx: ContextTypes.DEFAULT_TYPE):
    s = get_state(update.effective_user.id)
    mode = '🖼 Изображение' if s['mode'] == 'image' else '🎬 Видео'
    duration = s['video_clips'] * FRAMES_CLIP / FPS
    await update.message.reply_text(
        f'ℹ️ *Настройки:*\n'
        f'Режим: {mode}\n'
        f'Стиль: {STYLES[s["style"]][0]}\n'
        f'Шаги: {s["steps"]} | CFG: {s["cfg"]}\n'
        f'Длина видео: ~{duration:.0f} сек ({s["video_clips"]} клип(а))',
        parse_mode='Markdown'
    )

async def generate(update: Update, ctx: ContextTypes.DEFAULT_TYPE):
    uid = update.effective_user.id
    s = get_state(uid)
    prompt_ru = update.message.text.strip()
    prompt_en = translate_to_en(prompt_ru)
    mode = s['mode']
    style_name, style_prompt = STYLES[s['style']]

    if mode == 'video':
        duration = s['video_clips'] * FRAMES_CLIP / FPS
        wait_text = (
            f'🎬 Генерирую видео ~{duration:.0f} сек...\n'
            f'📝 _{prompt_ru}_\n'
            f'⏳ {s["video_clips"]} клип(а) × 3-5 мин = подожди'
        )
    else:
        wait_text = (
            f'🖼 Генерирую изображение...\n'
            f'📝 _{prompt_ru}_\n'
            f'🎨 {style_name} | ⏱ 1-2 мин'
        )

    wait_msg = await update.message.reply_text(wait_text, parse_mode='Markdown')

    try:
        if mode == 'image':
            result = image_pipe(
                prompt=f'{prompt_en}, {style_prompt}',
                negative_prompt='blurry, ugly, bad quality, watermark, text, deformed',
                num_inference_steps=s['steps'],
                guidance_scale=s['cfg'],
            ).images[0]
            path = f'/content/outputs/{uid}.png'
            result.save(path)
            await wait_msg.delete()
            with open(path, 'rb') as f:
                await update.message.reply_photo(
                    photo=f,
                    caption=f'✅ *{prompt_ru}*\n{style_name}',
                    parse_mode='Markdown'
                )
        else:
            path = f'/content/outputs/{uid}.mp4'
            generate_long_video(
                prompt=prompt_en,
                negative_prompt='blurry, bad quality, watermark, text, flickering',
                num_clips=s['video_clips'],
                frames_per_clip=FRAMES_CLIP,
                fps=FPS,
                output_path=path
            )
            await wait_msg.delete()
            duration = s['video_clips'] * FRAMES_CLIP / FPS
            with open(path, 'rb') as f:
                await update.message.reply_video(
                    video=f,
                    caption=f'✅ *{prompt_ru}* | {duration:.0f} сек',
                    parse_mode='Markdown'
                )
    except Exception as e:
        await wait_msg.edit_text(f'❌ Ошибка: {str(e)[:200]}')
        gc.collect()
        torch.cuda.empty_cache()

print('🤖 Запускаю бота...')
app = Application.builder().token(BOT_TOKEN).build()
app.add_handler(CommandHandler('start',    start))
app.add_handler(CommandHandler('image',    set_image))
app.add_handler(CommandHandler('video',    set_video))
app.add_handler(CommandHandler('style',    style_menu))
app.add_handler(CommandHandler('settings', settings_menu))
app.add_handler(CommandHandler('duration', duration_menu))
app.add_handler(CommandHandler('status',   status))
app.add_handler(CallbackQueryHandler(style_callback,    pattern='^style_'))
app.add_handler(CallbackQueryHandler(settings_callback, pattern='^(steps_|cfg_)'))
app.add_handler(CallbackQueryHandler(duration_callback, pattern='^clips_'))
app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, generate))
print('✅ Бот запущен! Открой его в Telegram и напиши /start')
print('⛔ Для остановки: Среда выполнения → Прервать выполнение')
app.run_polling(allowed_updates=Update.ALL_TYPES)